In [ ]:
!pip install -q sentencepiece

In [ ]:
# ============================================================
# 02. Import libraries
# ============================================================

import os
import random
import json

import numpy as np
import pandas as pd
import torch

import sentencepiece as spm

from datasets import Dataset, DatasetDict

from transformers import (
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

In [ ]:
# ============================================================
# 03. Set random seeds
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

In [ ]:
# ============================================================
# 04. Experiment configuration
# ============================================================

MODEL_NAME = "ThisenEkanayake/HelaBERT" # or "ThisenEkanayake/HelaBERT_Large"

# The actual pretrained BERT files are inside the
# HelaBERT/ subdirectory of the Hugging Face repository.
MODEL_SUBFOLDER = "HelaBERT"

TEXT_COLUMN = "noisy_text"
LABEL_COLUMN = "label"

NUM_LABELS = 2

MAX_LENGTH = 128

# checkpoint saving directory
OUTPUT_DIR = ""

# Trained model saving directory
FINAL_MODEL_DIR = ""

# Final results saving path (should end with .csv)
RESULTS_PATH = ""

print("Model:", MODEL_NAME)
print("Model subfolder:", MODEL_SUBFOLDER)
print("Maximum sequence length:", MAX_LENGTH)
print("Number of labels:", NUM_LABELS)

In [ ]:
# ============================================================
# 05. Dataset paths
# ============================================================

# Train and test datasets of the domain the model is trained on
TRAIN_PATH = ""
TEST_PATH = ""

# Test datasets of the domains the model is evaluated on
TEST_CLEAN_SINHALA = ""
TEST_TRANSLITERATION = ""
TEST_CODE_MIX = ""
TEST_MISSPELL = ""
TEST_COMPOUND = ""

In [ ]:
# ============================================================
# 06. Load JSONL datasets
# ============================================================

train_df = pd.read_json(
    TRAIN_PATH,
    lines=True
)

test_df = pd.read_json(
    TEST_PATH,
    lines=True
)

test_transliteration_df = pd.read_json(
    TEST_TRANSLITERATION,
    lines=True
)

test_codemix_df = pd.read_json(
    TEST_CODE_MIX,
    lines=True
)

test_clean_sinhala_df = pd.read_json(
    TEST_CLEAN_SINHALA,
    lines=True
)

test_misspell_df = pd.read_json(
    TEST_MISSPELL,
    lines=True
)

test_compound_df = pd.read_json(
    TEST_COMPOUND,
    lines=True
)

print("Original train:", train_df.shape)
print("Test:", test_df.shape)

print(
    "Transliteration:",
    test_transliteration_df.shape
)

print(
    "Code-mix:",
    test_codemix_df.shape
)

print(
    "Clean Sinhala:",
    test_clean_sinhala_df.shape
)

print(
    "Misspelling:",
    test_misspell_df.shape
)

print(
    "Compound:",
    test_compound_df.shape
)

In [ ]:
# ============================================================
# 07. Standardize text column
# ============================================================

# Main clean Sinhala files use "text".
train_df = train_df.rename(
    columns={"text": TEXT_COLUMN}
)

test_df = test_df.rename(
    columns={"text": TEXT_COLUMN}
)

test_clean_sinhala_df = (
    test_clean_sinhala_df.rename(
        columns={"text": TEXT_COLUMN}
    )
)

print(
    "Training columns:",
    train_df.columns.tolist()
)

In [ ]:
# ============================================================
# 08. Train / validation split
# ============================================================

# Original 7,500 training samples:
#
# 90% -> 6,750 training
# 10% ->   750 validation
#
# The 2,500 original test samples remain untouched.

train_df, valid_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df[LABEL_COLUMN]
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

print("\nTrain distribution:")
print(
    train_df[LABEL_COLUMN].value_counts()
)

print("\nValidation distribution:")
print(
    valid_df[LABEL_COLUMN].value_counts()
)

print("\nTest distribution:")
print(
    test_df[LABEL_COLUMN].value_counts()
)

In [ ]:
# ============================================================
# 09. Keep required columns
# ============================================================

dataframes = [
    train_df,
    valid_df,
    test_df,
    test_transliteration_df,
    test_codemix_df,
    test_clean_sinhala_df,
    test_misspell_df,
    test_compound_df,
]

for df in dataframes:

    # Keep only text and label.
    # Some datasets may have additional metadata.
    pass


train_df = train_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

valid_df = valid_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

test_df = test_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

test_transliteration_df = (
    test_transliteration_df[
        [TEXT_COLUMN, LABEL_COLUMN]
    ].copy()
)

test_codemix_df = (
    test_codemix_df[
        [TEXT_COLUMN, LABEL_COLUMN]
    ].copy()
)

test_clean_sinhala_df = (
    test_clean_sinhala_df[
        [TEXT_COLUMN, LABEL_COLUMN]
    ].copy()
)

test_misspell_df = (
    test_misspell_df[
        [TEXT_COLUMN, LABEL_COLUMN]
    ].copy()
)

test_compound_df = (
    test_compound_df[
        [TEXT_COLUMN, LABEL_COLUMN]
    ].copy()
)


# Remove missing values.
for df in [
    train_df,
    valid_df,
    test_df,
    test_transliteration_df,
    test_codemix_df,
    test_clean_sinhala_df,
    test_misspell_df,
    test_compound_df,
]:

    df.dropna(
        subset=[
            TEXT_COLUMN,
            LABEL_COLUMN
        ],
        inplace=True
    )

    df[TEXT_COLUMN] = (
        df[TEXT_COLUMN].astype(str)
    )

    df[LABEL_COLUMN] = (
        df[LABEL_COLUMN].astype(int)
    )


print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

In [ ]:
# ============================================================
# 10. Label mapping
# ============================================================

label2id = {
    "non-hate": 0,
    "hate": 1,
}

id2label = {
    0: "non-hate",
    1: "hate",
}


# Create Hugging Face "labels" column.
train_df["labels"] = train_df[LABEL_COLUMN]
valid_df["labels"] = valid_df[LABEL_COLUMN]
test_df["labels"] = test_df[LABEL_COLUMN]

test_transliteration_df["labels"] = (
    test_transliteration_df[LABEL_COLUMN]
)

test_codemix_df["labels"] = (
    test_codemix_df[LABEL_COLUMN]
)

test_clean_sinhala_df["labels"] = (
    test_clean_sinhala_df[LABEL_COLUMN]
)

test_misspell_df["labels"] = (
    test_misspell_df[LABEL_COLUMN]
)

test_compound_df["labels"] = (
    test_compound_df[LABEL_COLUMN]
)


print("label2id:", label2id)
print("id2label:", id2label)

In [ ]:
# ============================================================
# 11. Convert to Hugging Face Dataset
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df[
        [TEXT_COLUMN, "labels"]
    ],
    preserve_index=False
)

valid_dataset = Dataset.from_pandas(
    valid_df[
        [TEXT_COLUMN, "labels"]
    ],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[
        [TEXT_COLUMN, "labels"]
    ],
    preserve_index=False
)


dataset = DatasetDict({

    "train":
        train_dataset,

    "validation":
        valid_dataset,

    "test":
        test_dataset,
})


# Five external evaluation datasets.

test_transliteration_dataset = (
    Dataset.from_pandas(
        test_transliteration_df[
            [TEXT_COLUMN, "labels"]
        ],
        preserve_index=False
    )
)

test_codemix_dataset = (
    Dataset.from_pandas(
        test_codemix_df[
            [TEXT_COLUMN, "labels"]
        ],
        preserve_index=False
    )
)

test_clean_sinhala_dataset = (
    Dataset.from_pandas(
        test_clean_sinhala_df[
            [TEXT_COLUMN, "labels"]
        ],
        preserve_index=False
    )
)

test_misspell_dataset = (
    Dataset.from_pandas(
        test_misspell_df[
            [TEXT_COLUMN, "labels"]
        ],
        preserve_index=False
    )
)

test_compound_dataset = (
    Dataset.from_pandas(
        test_compound_df[
            [TEXT_COLUMN, "labels"]
        ],
        preserve_index=False
    )
)


other_test_dataset = DatasetDict({

    "transliteration_test":
        test_transliteration_dataset,

    "codemix_test":
        test_codemix_dataset,

    "clean_sinhala_test":
        test_clean_sinhala_dataset,

    "misspell_test":
        test_misspell_dataset,

    "compound_test":
        test_compound_dataset,
})


print(dataset)
print(other_test_dataset)

In [ ]:
# ============================================================
# 12. Download HelaBERT SentencePiece tokenizer
# ============================================================

from huggingface_hub import hf_hub_download

TOKENIZER_PATH = hf_hub_download(
    repo_id=MODEL_NAME,
    filename="tokenizer/unigram_32000_0.9995.model"
)

print(
    "Tokenizer downloaded to:"
)

print(TOKENIZER_PATH)

In [ ]:
# ============================================================
# 13. Load SentencePiece tokenizer
# ============================================================

sp = spm.SentencePieceProcessor()

sp.load(
    TOKENIZER_PATH
)

print(
    "Vocabulary size:",
    sp.get_piece_size()
)

In [ ]:
# ============================================================
# 14. Inspect special tokens
# ============================================================

special_tokens = [
    "[PAD]",
    "[UNK]",
    "[CLS]",
    "[SEP]",
    "[MASK]",
]

for token in special_tokens:

    token_id = sp.piece_to_id(token)

    print(
        f"{token:8s} -> {token_id}"
    )

In [ ]:
# ============================================================
# 15. Inspect HelaBERT tokenization
# ============================================================

sample_texts = (
    train_df[TEXT_COLUMN]
    .head(10)
    .tolist()
)

for text in sample_texts:

    ids = sp.encode(
        text,
        out_type=int
    )

    pieces = sp.id_to_piece(ids)

    print("=" * 80)

    print("TEXT:")
    print(text)

    print("\nTOKENS:")
    print(pieces)

    print("\nNUMBER OF TOKENS:")
    print(len(ids))

In [ ]:
# ============================================================
# 16. HelaBERT tokenization function
# ============================================================

PAD_ID = sp.piece_to_id("[PAD]")
UNK_ID = sp.piece_to_id("[UNK]")
CLS_ID = sp.piece_to_id("[CLS]")
SEP_ID = sp.piece_to_id("[SEP]")

print("PAD:", PAD_ID)
print("UNK:", UNK_ID)
print("CLS:", CLS_ID)
print("SEP:", SEP_ID)


def tokenize_text(text):

    # Encode Sinhala text into SentencePiece IDs.
    token_ids = sp.encode(
        text,
        out_type=int
    )

    # Reserve two positions for [CLS] and [SEP].
    token_ids = token_ids[
        :MAX_LENGTH - 2
    ]

    # Add BERT special tokens.
    input_ids = (
        [CLS_ID]
        + token_ids
        + [SEP_ID]
    )

    attention_mask = [
        1
        for _ in input_ids
    ]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
    }

In [ ]:
# ============================================================
# 17. Tokenize datasets
# ============================================================

def tokenize_function(examples):

    input_ids = []
    attention_masks = []

    for text in examples[TEXT_COLUMN]:

        result = tokenize_text(text)

        input_ids.append(
            result["input_ids"]
        )

        attention_masks.append(
            result["attention_mask"]
        )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
    }


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
)

tokenized_test_dataset = (
    other_test_dataset.map(
        tokenize_function,
        batched=True,
    )
)


print(tokenized_dataset)
print(tokenized_test_dataset)

In [ ]:
# ============================================================
# 18. Custom padding data collator
# ============================================================

class HelaBERTDataCollator:

    def __init__(self, pad_token_id):

        self.pad_token_id = pad_token_id


    def __call__(self, features):

        max_length = max(
            len(feature["input_ids"])
            for feature in features
        )

        batch_input_ids = []
        batch_attention_masks = []
        batch_labels = []

        for feature in features:

            input_ids = feature[
                "input_ids"
            ]

            attention_mask = feature[
                "attention_mask"
            ]

            padding_length = (
                max_length
                - len(input_ids)
            )

            input_ids = (
                input_ids
                + [
                    self.pad_token_id
                ] * padding_length
            )

            attention_mask = (
                attention_mask
                + [0] * padding_length
            )

            batch_input_ids.append(
                input_ids
            )

            batch_attention_masks.append(
                attention_mask
            )

            if "labels" in feature:

                batch_labels.append(
                    feature["labels"]
                )

        batch = {

            "input_ids": torch.tensor(
                batch_input_ids,
                dtype=torch.long
            ),

            "attention_mask": torch.tensor(
                batch_attention_masks,
                dtype=torch.long
            ),
        }

        if batch_labels:

            batch["labels"] = torch.tensor(
                batch_labels,
                dtype=torch.long
            )

        return batch


data_collator = HelaBERTDataCollator(
    pad_token_id=PAD_ID
)

print("HelaBERT data collator created.")

In [ ]:
# ============================================================
# 19. Load HelaBERT sequence classification model
# ============================================================

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    subfolder=MODEL_SUBFOLDER,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

print("HelaBERT loaded.")

In [ ]:
# ============================================================
# 20. Inspect model
# ============================================================

print(model)

print(
    "\nHidden size:",
    model.config.hidden_size
)

print(
    "Number of layers:",
    model.config.num_hidden_layers
)

print(
    "Vocabulary size:",
    model.config.vocab_size
)

In [ ]:
# ============================================================
# 21. FREEZE HelaBERT ENCODER
# Classification head only fine-tuning
# ============================================================

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False


# Unfreeze ONLY the classification head
for param in model.classifier.parameters():
    param.requires_grad = True


# Check parameter counts
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.2f}%"
)

In [ ]:
# ============================================================
# 22. Verify trainable parameters
# ============================================================

for name, param in model.named_parameters():

    if param.requires_grad:

        print(
            "TRAINABLE:",
            name
        )

In [ ]:
# ============================================================
# 23. Evaluation metrics
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    # Accuracy
    accuracy = accuracy_score(
        labels,
        predictions
    )

    # Macro precision, recall, F1
    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    # Weighted F1
    _, _, weighted_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {

        "accuracy":
            accuracy,

        "macro_precision":
            precision,

        "macro_recall":
            recall,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,
    }

In [ ]:
# ============================================================
# 24. Training configuration
# ============================================================

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    num_train_epochs=5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=32,

    # Classification head-only fine-tuning learning rate.
    learning_rate=1e-3,

    weight_decay=0.01,


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,


    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    logging_strategy="steps",

    logging_steps=50,


    # --------------------------------------------------------
    # GPU
    # --------------------------------------------------------

    fp16=torch.cuda.is_available(),


    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    seed=SEED,


    # --------------------------------------------------------
    # Checkpoints
    # --------------------------------------------------------

    save_total_limit=2,

    report_to="none",
)


print("Training configuration created.")

In [ ]:
# ============================================================
# 25. Create Trainer
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=(
        tokenized_dataset["train"]
    ),

    eval_dataset=(
        tokenized_dataset["validation"]
    ),

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

print("Trainer created.")

In [ ]:
# ============================================================
# 26. Train HelaBERT
# ============================================================

train_result = trainer.train()

print(
    "\nTraining completed."
)

In [ ]:
# ============================================================
# 27. Validation evaluation
# ============================================================

validation_results = trainer.evaluate(
    tokenized_dataset["validation"]
)

print("\nValidation results:")

for key, value in validation_results.items():

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 28. Main test evaluation
# ============================================================

test_results = trainer.evaluate(
    tokenized_dataset["test"]
)

print("\nMain Test Results:")

for key, value in test_results.items():

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 29. Transliteration evaluation
# ============================================================

test_results_on_transliteration = (
    trainer.evaluate(
        tokenized_test_dataset[
            "transliteration_test"
        ]
    )
)

print(
    "\nTest on Transliteration:"
)

for key, value in (
    test_results_on_transliteration.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 30. Code-mix evaluation
# ============================================================

test_results_on_codemix = (
    trainer.evaluate(
        tokenized_test_dataset[
            "codemix_test"
        ]
    )
)

print(
    "\nTest on Code-Mix:"
)

for key, value in (
    test_results_on_codemix.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 31. Clean Sinhala evaluation
# ============================================================

test_results_on_clean = (
    trainer.evaluate(
        tokenized_test_dataset[
            "clean_sinhala_test"
        ]
    )
)

print(
    "\nTest on Clean Sinhala:"
)

for key, value in (
    test_results_on_clean.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 32. Misspelling evaluation
# ============================================================

test_results_on_misspell = (
    trainer.evaluate(
        tokenized_test_dataset[
            "misspell_test"
        ]
    )
)

print(
    "\nTest on Misspelling:"
)

for key, value in (
    test_results_on_misspell.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 33. Compound evaluation
# ============================================================

test_results_on_compound = (
    trainer.evaluate(
        tokenized_test_dataset[
            "compound_test"
        ]
    )
)

print(
    "\nTest on Compound:"
)

for key, value in (
    test_results_on_compound.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 34. Classification report - Main Test
# ============================================================

predictions = trainer.predict(
    tokenized_dataset["test"]
)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=-1
)

true_labels = predictions.label_ids


print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=[
            id2label[0],
            id2label[1],
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 35. Classification report - Transliteration
# ============================================================

predictions_tr = trainer.predict(
    tokenized_test_dataset[
        "transliteration_test"
    ]
)

predicted_labels_tr = np.argmax(
    predictions_tr.predictions,
    axis=-1
)

true_labels_tr = (
    predictions_tr.label_ids
)


print(
    classification_report(
        true_labels_tr,
        predicted_labels_tr,
        target_names=[
            id2label[0],
            id2label[1],
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 36. Classification report - Code-Mix
# ============================================================

predictions_cm = trainer.predict(
    tokenized_test_dataset[
        "codemix_test"
    ]
)

predicted_labels_cm = np.argmax(
    predictions_cm.predictions,
    axis=-1
)

true_labels_cm = (
    predictions_cm.label_ids
)


print(
    classification_report(
        true_labels_cm,
        predicted_labels_cm,
        target_names=[
            id2label[0],
            id2label[1],
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 37. Classification report - Clean Sinhala
# ============================================================

predictions_cs = trainer.predict(
    tokenized_test_dataset[
        "clean_sinhala_test"
    ]
)

predicted_labels_cs = np.argmax(
    predictions_cs.predictions,
    axis=-1
)

true_labels_cs = (
    predictions_cs.label_ids
)


print(
    classification_report(
        true_labels_cs,
        predicted_labels_cs,
        target_names=[
            id2label[0],
            id2label[1],
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 38. Classification report - Misspelling
# ============================================================

predictions_misspell = trainer.predict(
    tokenized_test_dataset[
        "misspell_test"
    ]
)

predicted_labels_misspell = np.argmax(
    predictions_misspell.predictions,
    axis=-1
)

true_labels_misspell = (
    predictions_misspell.label_ids
)


print(
    classification_report(
        true_labels_misspell,
        predicted_labels_misspell,
        target_names=[
            id2label[0],
            id2label[1],
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 39. Classification report - Compound
# ============================================================

predictions_compound = trainer.predict(
    tokenized_test_dataset[
        "compound_test"
    ]
)

predicted_labels_compound = np.argmax(
    predictions_compound.predictions,
    axis=-1
)

true_labels_compound = (
    predictions_compound.label_ids
)


print(
    classification_report(
        true_labels_compound,
        predicted_labels_compound,
        target_names=[
            id2label[0],
            id2label[1],
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 40. Confusion matrix
# ============================================================

cm = confusion_matrix(
    true_labels,
    predicted_labels
)

print(
    "Confusion Matrix:"
)

print(cm)

In [ ]:
# ============================================================
# 41. Compare all five evaluation conditions
# ============================================================

comparison_df = pd.DataFrame({

    "Dataset": [

        "Main Test",

        "Transliteration",

        "Code-Mix",

        "Clean Sinhala",

        "Misspelling",

        "Compound",
    ],

    "Accuracy": [

        test_results[
            "eval_accuracy"
        ],

        test_results_on_transliteration[
            "eval_accuracy"
        ],

        test_results_on_codemix[
            "eval_accuracy"
        ],

        test_results_on_clean[
            "eval_accuracy"
        ],

        test_results_on_misspell[
            "eval_accuracy"
        ],

        test_results_on_compound[
            "eval_accuracy"
        ],
    ],

    "Macro Precision": [

        test_results[
            "eval_macro_precision"
        ],

        test_results_on_transliteration[
            "eval_macro_precision"
        ],

        test_results_on_codemix[
            "eval_macro_precision"
        ],

        test_results_on_clean[
            "eval_macro_precision"
        ],

        test_results_on_misspell[
            "eval_macro_precision"
        ],

        test_results_on_compound[
            "eval_macro_precision"
        ],
    ],

    "Macro Recall": [

        test_results[
            "eval_macro_recall"
        ],

        test_results_on_transliteration[
            "eval_macro_recall"
        ],

        test_results_on_codemix[
            "eval_macro_recall"
        ],

        test_results_on_clean[
            "eval_macro_recall"
        ],

        test_results_on_misspell[
            "eval_macro_recall"
        ],

        test_results_on_compound[
            "eval_macro_recall"
        ],
    ],

    "Macro F1": [

        test_results[
            "eval_macro_f1"
        ],

        test_results_on_transliteration[
            "eval_macro_f1"
        ],

        test_results_on_codemix[
            "eval_macro_f1"
        ],

        test_results_on_clean[
            "eval_macro_f1"
        ],

        test_results_on_misspell[
            "eval_macro_f1"
        ],

        test_results_on_compound[
            "eval_macro_f1"
        ],
    ],

    "Weighted F1": [

        test_results[
            "eval_weighted_f1"
        ],

        test_results_on_transliteration[
            "eval_weighted_f1"
        ],

        test_results_on_codemix[
            "eval_weighted_f1"
        ],

        test_results_on_clean[
            "eval_weighted_f1"
        ],

        test_results_on_misspell[
            "eval_weighted_f1"
        ],

        test_results_on_compound[
            "eval_weighted_f1"
        ],
    ],
})


display(comparison_df)

In [ ]:
# ============================================================
# 42. Save trained HelaBERT
# ============================================================

trainer.save_model(
    FINAL_MODEL_DIR
)

print(
    "Model saved to:"
)

print(FINAL_MODEL_DIR)

In [ ]:
import shutil

final_tokenizer_path = os.path.join(
    FINAL_MODEL_DIR,
    "unigram_32000_0.9995.model"
)

shutil.copy(
    TOKENIZER_PATH,
    final_tokenizer_path
)

print(
    "SentencePiece tokenizer saved to:"
)

print(final_tokenizer_path)

In [ ]:
# ============================================================
# 43. Save experiment results
# ============================================================

experiment_results = {

    "experiment":
        "HelaBERT Classification Head-Only Fine Tuning",

    "model":
        MODEL_NAME,

    "model_subfolder":
        MODEL_SUBFOLDER,

    "tokenizer":
        "SentencePiece Unigram 32000",

    "encoder_trainable":
        True,

    "classification_head_trainable":
        True,

    "max_length":
        MAX_LENGTH,

    "seed":
        SEED,

    "train_samples":
        len(train_df),

    "validation_samples":
        len(valid_df),

    "test_samples":
        len(test_df),

    "accuracy":
        test_results.get(
            "eval_accuracy"
        ),

    "macro_precision":
        test_results.get(
            "eval_macro_precision"
        ),

    "macro_recall":
        test_results.get(
            "eval_macro_recall"
        ),

    "macro_f1":
        test_results.get(
            "eval_macro_f1"
        ),

    "weighted_f1":
        test_results.get(
            "eval_weighted_f1"
        ),

    "transliteration_macro_f1":
        test_results_on_transliteration.get(
            "eval_macro_f1"
        ),

    "codemix_macro_f1":
        test_results_on_codemix.get(
            "eval_macro_f1"
        ),

    "clean_sinhala_macro_f1":
        test_results_on_clean.get(
            "eval_macro_f1"
        ),

    "misspell_macro_f1":
        test_results_on_misspell.get(
            "eval_macro_f1"
        ),

    "compound_macro_f1":
        test_results_on_compound.get(
            "eval_macro_f1"
        ),

    "total_parameters":
        total_params,

    "trainable_parameters":
        trainable_params,

    "trainable_percentage":
        (
            100
            * trainable_params
            / total_params
        ),
}


results_df = pd.DataFrame(
    [experiment_results]
)

display(results_df)

In [ ]:
# ============================================================
# 44. Save results CSV
# ============================================================

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print(
    "Results saved to:"
)

print(RESULTS_PATH)

In [ ]:
# ============================================================
# 45. Final experiment summary
# ============================================================

print("=" * 70)

print(
    "HELABERT FULL FINE-TUNING"
)

print("=" * 70)

print(
    "\nModel:",
    MODEL_NAME
)

print(
    "Parameters:",
    f"{total_params:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)

print(
    "Max length:",
    MAX_LENGTH
)

print(
    "\nMain Test Macro-F1:",
    test_results[
        "eval_macro_f1"
    ]
)

print(
    "Transliteration Macro-F1:",
    test_results_on_transliteration[
        "eval_macro_f1"
    ]
)

print(
    "Code-Mix Macro-F1:",
    test_results_on_codemix[
        "eval_macro_f1"
    ]
)

print(
    "Clean Sinhala Macro-F1:",
    test_results_on_clean[
        "eval_macro_f1"
    ]
)

print(
    "Misspelling Macro-F1:",
    test_results_on_misspell[
        "eval_macro_f1"
    ]
)

print(
    "Compound Macro-F1:",
    test_results_on_compound[
        "eval_macro_f1"
    ]
)

print("=" * 70)